# 11_Retrieval.ipynb
=================
This notebook demonstrates the semantic retrieval pipeline using MiniLM embeddings and FAISS vector indexing. We construct a semantic index from our support ticket corpus, run similarity queries, and compute standard information retrieval metrics (Recall@K, MRR, MAP).

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT).exists():
    REPO_ROOT = REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pathlib import Path
import pandas as pd
import numpy as np

from src.data.dataset import load_and_preprocess_dataset
from src.models.transformer.retrieval import SemanticRetriever
from src.utils.constants import OUTPUT_DIR

## 1. Load Dataset Corpus
We load our preprocessed customer support ticket splits.

In [ ]:
splits = load_and_preprocess_dataset()
test_df = splits["test"]
corpus = test_df["text"].tolist()
print(f"Corpus size: {len(corpus)} tickets")

## 2. Build FAISS Index
We initialize our `SemanticRetriever` and construct the index using Cosine Similarity.

In [ ]:
retriever = SemanticRetriever(metric="cosine")
retriever.build_index(corpus)

## 3. Perform Similarity Search
We run a query to locate semantically similar tickets from the corpus.

In [ ]:
query = "My account is locked due to wrong passcode attempt"
results = retriever.retrieve(query, top_k=3)

for res in results:
    print(f"Rank {res['rank']} (Score: {res['score']:.4f}) [Index: {res['index']}]: {res['text']}")

## 4. Evaluate Retrieval Quality
We define queries and use matching labels to build ground-truth indices. We then evaluate Recall@K, MRR, and MAP.

In [ ]:
# Select top 10 tickets as validation queries
val_queries = test_df["text"].iloc[:10].tolist()
val_labels = test_df["label"].iloc[:10].tolist()

# Ground truth: other documents sharing the same category/label
ground_truth_indices = []
for label in val_labels:
    matching_ids = test_df[test_df["label"] == label].index.tolist()
    ground_truth_indices.append(matching_ids)

# Run evaluation
metrics = retriever.evaluate_retrieval(val_queries, ground_truth_indices, k=5)
for name, value in metrics.items():
    print(f"{name}: {value:.4%}")

## 5. Serialise Retrieval Database
We save the index and corpus metadata to the disk.

In [ ]:
index_dir = OUTPUT_DIR / "retrieval_index"
retriever.save_index(index_dir)
print(f"Index saved to: {index_dir}")

In [ ]:
# Export Phase Manifest
from src.utils.artifacts import save_yaml
from src.api.app import get_git_commit

manifest = {
    "phase": "11_Retrieval",
    "retrieval_status": "FAISS index saved and verified",
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_11_retrieval.yaml")
print("YAML manifest saved successfully:")
print(manifest)
